In [ ]:
"""
monitor_weekly.py -- weekly reliability monitor for the deployed hazard model.
Doubles as HGT instrumentation: a de-calibration of this layer is a measured
condition-set transition for this D(n).

RULES
  M.1  Alert bands are empirical: MODE="baseline" computes every metric per
       ISO-week over the deployed model's holdout parquet and stores p5/p50/p95.
       Weekly runs are judged against those, not against opinions.
  M.2  Decile cuts are frozen from the holdout p_cal distribution at baseline
       time. Weekly tables always bin on the frozen cuts.
  M.3  The weekly run replays the week through the DEPLOYED Engine+Scorer
       (hazard_worker) -- the live code path -- so every week is also a
       regression test of the worker. Outcomes are computed independently from
       JMA sign flips (Stage-0 definition).
  M.4  Data sanity runs before any model metric: session bar counts, duplicate
       timestamps, NaNs. Sanity FAIL aborts the report.
  M.5  Verdicts: ALERT if weekly skill < p5, or top-bin realized < p5, or
       GREEN-zone break rate > p95. WATCH if any other tracked metric leaves
       its band. Two consecutive WATCHes on the same metric = treat as ALERT.
       Action ladder on ALERT: (1) recheck data prep, (2) eyeball worst days in
       the day plot, (3) second consecutive ALERT -> refit under the identical
       audit, paired-compare, deploy if better. Always log the dates.
  M.6  One summary row per run appends to monitoring_log.parquet.
"""

import json
import numpy as np
import pandas as pd

from hazard_worker import (Engine, Scorer, load_research_frame,
                           FRAME, WARMUP_BARS, MODEL_PATH,
                           GREEN_MAX, RED_MIN)

# ---------------------------------------------------------------- CONFIG
MODE = "baseline"                    # "baseline" | "week"
PRED_PATH = "data/stage-5/pred_lgbm5b_5c2raw_01-2025_6s.parquet"
BASELINE_PATH = "data/monitoring/baseline.json"
LOG_PATH = "data/monitoring/monitoring_log.parquet"
WEEK_FROM = None                     # "YYYY-MM-DD" or None -> last 5 sessions
WEEK_TO = None
EXPECTED_SESSION_BARS = 3900
# ----------------------------------------------------------------

TRACKED = ["skill", "top_realized", "green_break", "mono_viol",
           "red_frac", "red_precision", "ll_gap", "base_rate"]
ALERT_LOW = {"skill", "top_realized"}
ALERT_HIGH = {"green_break"}


def signfill_targets(jma):
    """Stage-0 target definition: sign-of-diff with zero-carry, flip = target."""
    d = np.zeros(len(jma))
    d[1:] = np.diff(jma)
    s = np.sign(d).astype(np.int8)
    pos = np.maximum.accumulate(np.where(s != 0, np.arange(len(s)), -1))
    out = np.zeros(len(s), np.int8)
    ok = pos >= 0
    out[ok] = s[pos[ok]]
    tgt = np.zeros(len(jma), bool)
    tgt[1:] = (out[1:] != out[:-1]) & (out[1:] != 0) & (out[:-1] != 0)
    return tgt


def ll(y, p):
    p = np.clip(np.asarray(p, np.float64), 1e-12, 1 - 1e-12)
    return float(-np.mean(y * np.log(p) + (1 - y) * np.log(1 - p)))


def metrics(y, p, p_cal, cuts):
    y = np.asarray(y, np.float64)
    r = y.mean()
    m = {"n": int(len(y)), "base_rate": float(r)}
    m["ll_pcal"] = ll(y, p_cal)
    m["ll_const"] = ll(y, np.full_like(y, r))
    m["skill"] = 1.0 - m["ll_pcal"] / m["ll_const"]
    m["ll_gap"] = ll(y, p) - m["ll_pcal"]          # <0 sustained => iso aging

    b = np.searchsorted(cuts, p_cal)
    tbl = pd.DataFrame({"bin": b, "p": p_cal, "y": y}).groupby("bin").agg(
        n=("y", "size"), mean_p=("p", "mean"), realized=("y", "mean"))
    tbl = tbl.reindex(range(len(cuts) + 1))
    rr = tbl["realized"].to_numpy()
    valid = tbl["n"].to_numpy() >= 200
    dif = np.diff(np.where(valid, rr, np.nan))
    m["mono_viol"] = int(np.nansum(dif < 0))
    m["top_realized"] = float(rr[-1]) if valid[-1] else np.nan
    green = p_cal < GREEN_MAX
    m["green_break"] = float(y[green].mean()) if green.any() else np.nan
    red = p_cal >= RED_MIN
    m["red_frac"] = float(red.mean())
    m["red_precision"] = float(y[red].mean()) if red.any() else np.nan
    return m, tbl


# ---------------------------------------------------------------- baseline

def build_baseline():
    pred = pd.read_parquet(PRED_PATH)
    cuts = np.quantile(pred["p_cal"], np.arange(0.1, 1.0, 0.1)).tolist()  # M.2
    iso = pred["timestamp"].dt.isocalendar()
    pred["wk"] = iso["year"].astype(str) + "-" + iso["week"].astype(str)

    rows = []
    for wk, g in pred.groupby("wk"):
        m, _ = metrics(g["is_target"].to_numpy(), g["p"].to_numpy(),
                       g["p_cal"].to_numpy(), cuts)
        m["wk"] = wk
        rows.append(m)
    hist = pd.DataFrame(rows)

    bands = {k: {"p5": float(np.nanpercentile(hist[k], 5)),
                 "p50": float(np.nanpercentile(hist[k], 50)),
                 "p95": float(np.nanpercentile(hist[k], 95))}
             for k in TRACKED}
    base = {"cuts": cuts, "bands": bands, "n_weeks": int(len(hist)),
            "pred_path": PRED_PATH, "green_max": GREEN_MAX, "red_min": RED_MIN}
    with open(BASELINE_PATH, "w") as f:
        json.dump(base, f, indent=2)
    print(json.dumps(bands, indent=2))
    print(f"baseline over {len(hist)} weeks -> {BASELINE_PATH}")
    return base, hist


# ---------------------------------------------------------------- weekly

def sanity(df):                                                            # M.4
    ok = True
    sizes = df.groupby(df["timestamp"].dt.date).size()
    for d, n in sizes.items():
        if n > EXPECTED_SESSION_BARS:
            print(f"SANITY FAIL: {d} has {n} bars (> {EXPECTED_SESSION_BARS})")
            ok = False
        elif n != EXPECTED_SESSION_BARS:
            print(f"note: {d} short session ({n} bars) -- early close?")
    if df["timestamp"].duplicated().any():
        print("SANITY FAIL: duplicate timestamps")
        ok = False
    cols = ["rawOpen", "rawLast", "JMA", "jmaD1", "jmaD2",
            "tickJmaD1", "tickJmaD2"]
    if df[cols].isna().any().any():
        print("SANITY FAIL: NaNs in feed columns")
        ok = False
    return ok


def replay_week(df, scorer):                                               # M.3
    cols = ["rawOpen", "rawLast", "JMA", "jmaD1", "jmaD2",
            "tickJmaD1", "tickJmaD2"]
    rows, ys, tss = [], [], []
    for day, g in df.groupby(df["timestamp"].dt.date, sort=True):
        eng = Engine()
        V = g[cols].to_numpy(np.float64)
        tgt = signfill_targets(g["JMA"].to_numpy(np.float64))
        n = len(g)
        for i in range(n):
            t, feats = eng.ingest(*V[i])
            if feats is not None and t < n:
                rows.append(feats)
                ys.append(tgt[t])
                tss.append(g["timestamp"].iloc[t])
    X = np.vstack(rows)
    p, p_cal = scorer.predict(X)
    return pd.DataFrame({"timestamp": tss, "y": np.array(ys, np.int8),
                         "p": p.astype(np.float32),
                         "p_cal": p_cal.astype(np.float32)})


def verdict(m, bands):                                                     # M.5
    alerts, watches = [], []
    for k in TRACKED:
        v = m.get(k, np.nan)
        if not np.isfinite(v):
            continue
        lo, hi = bands[k]["p5"], bands[k]["p95"]
        if k in ALERT_LOW and v < lo:
            alerts.append(f"{k}={v:.4g} < p5 {lo:.4g}")
        elif k in ALERT_HIGH and v > hi:
            alerts.append(f"{k}={v:.4g} > p95 {hi:.4g}")
        elif v < lo or v > hi:
            watches.append(f"{k}={v:.4g} outside [{lo:.4g}, {hi:.4g}]")
    if alerts:
        return "ALERT", alerts + watches
    if watches:
        return "WATCH", watches
    return "OK", []


def run_week():
    with open(BASELINE_PATH) as f:
        base = json.load(f)
    cuts = np.asarray(base["cuts"])

    df = load_research_frame()
    if WEEK_FROM is None:
        days = sorted(df["timestamp"].dt.date.unique())[-5:]
        df = df[df["timestamp"].dt.date.isin(days)]
    else:
        df = df[(df["timestamp"] >= WEEK_FROM) & (df["timestamp"] <= WEEK_TO)]
    print(f"window: {df.timestamp.min()} .. {df.timestamp.max()}  "
          f"({df.timestamp.dt.date.nunique()} sessions, {len(df)} bars)")

    if not sanity(df):
        print("VERDICT: DATA FAIL -- fix data prep, rerun. No model metrics read.")
        return None

    scorer = Scorer(MODEL_PATH)
    w = replay_week(df, scorer)
    m, tbl = metrics(w["y"].to_numpy(), w["p"].to_numpy(),
                     w["p_cal"].to_numpy(), cuts)
    v, notes = verdict(m, base["bands"])

    print(tbl.to_string())
    print(json.dumps(m, indent=2))
    print("VERDICT:", v)
    for s in notes:
        print("  -", s)

    row = dict(m)
    row.update({"verdict": v, "from": str(df.timestamp.min()),
                "to": str(df.timestamp.max()),
                "notes": "; ".join(notes)})
    try:
        log = pd.read_parquet(LOG_PATH)
        log = pd.concat([log, pd.DataFrame([row])], ignore_index=True)
    except FileNotFoundError:
        log = pd.DataFrame([row])
    log.to_parquet(LOG_PATH, index=False)                                  # M.6
    return w, m, tbl


# ---------------------------------------------------------------- usage

if __name__ == "__main__":
    if MODE == "baseline":
        base, hist = build_baseline()
    else:
        out = run_week()